In [1]:
from fit import *          # dataset, models, losses, train, paths (and utils via fit)
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [2]:
# Trajectory batches on the factored (lead, ch, x, y) grid: z_prev / z_next are (b, K, C, H, W), lead is (K,) = leads 2..24.
# x0-free: condition=False, so x0 in each batch is None. 80/20 train/val split, 8 trajectories per batch.
train_ds = TrajDataset(invert_euler960, split='train', device=device, traj_per_batch=8, condition=False)

# test code
lead, z_prev, z_next, _ = next(train_ds.iterate_batches())
print(train_ds.n, 'trajectories,', train_ds.size, 'loss terms')
print(lead.shape, z_prev.shape, z_next.shape)
print(f"lead max: {torch.max(lead)}, lead min {torch.min(lead)}")

ValueError: This file contains pickled (object) data. If you trust the file you can load it unsafely using the `allow_pickle=` keyword argument or `pickle.load()`.

In [ ]:
val_ds = TrajDataset(invert_euler960, split='val', device=device, traj_per_batch=8, condition=False)   # same data as train
print(val_ds.n, 'trajectories,', val_ds.size, 'loss terms')

240 trajectories, 45219840 loss terms


In [ ]:
# Tuned x0-free setting, c2 only:
# (lead, ch, y) MLP with 1 hidden layer of 256 units, lead -> positional embedding (64), ch -> nn.Embedding (64),
# x ignored (x_dim=0), y -> two Fourier modes cos/sin(2*pi*m*y/64), m=1,2 (y_dim=4).
# y_dim=4 beat y_dim=2: val 0.08715 vs 0.08722 at the last epoch (3 seeds each).
torch.manual_seed(0)
model = TimeVaryingOU(emb_dim=64, hidden_dim=256, n_hidden=1, x_dim=0, y_dim=4, fit_sigma=False)
print(sum(p.numel() for p in model.parameters()), 'parameters')

34433 parameters


In [ ]:
# Tuned x0-free training setting:
# AdamW lr 1e-3, weight decay 1e-4, cosine lr schedule stepped every batch over all 150 epochs, no early stopping.
history = train(model, train_ds, val_ds, c2_mle_without_x0, epochs=150, lr=1e-3, weight_decay=1e-4,
                cosine=False, patience=40)
plot_loss(history, result_path / 'c2_without_condition_loss.png', title='Without x0-conditioned time-varying OU')

epoch 1/150  train nll 0.0754  val nll 0.0880
epoch 10/150  train nll 0.0755  val nll 0.0871
epoch 20/150  train nll 0.0754  val nll 0.0877
epoch 30/150  train nll 0.0754  val nll 0.0872
epoch 40/150  train nll 0.0755  val nll 0.0874
epoch 50/150  train nll 0.0754  val nll 0.0874
early stop at epoch 52: no val improvement for 40 epochs
best val nll 0.08694 at epoch 12, last val nll 0.08701


In [ ]:
torch.save(history['best_state'], model_save_path / "c2_euler_withoutx0.pt")      # best-val state; use model.state_dict() for the last epoch
print('saved', save_path)

saved /home/trace/Documents/Projects/Thesis/Analysis/Models/c2_euler_withoutx0.pt
